In [1]:
import os

In [2]:
os.chdir("../")
%pwd

'd:\\AI Projects\\Thesis project'

In [3]:
from dataclasses import dataclass
from pathlib import Path
from typing import List

@dataclass(frozen=True)
class DataAnalysisConfig:
    root_dir: Path
    input_data_dir: Path
    reports_dir: Path
    target_variables: List[str]

In [4]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([Path(self.config.artifacts_root)])

    def get_data_analysis_config(self) -> DataAnalysisConfig:
        config = self.config.data_analysis
        schema = self.schema

        # 1. Dynamically build the target variables list from schema.yaml
        feature_columns = list(schema.columns.keys())
        target_column = schema.target_column.name
        
        # Combine and de-duplicate
        target_variables = list(set(feature_columns + [target_column]))

        # 2. Create the directories for analysis reports
        create_directories([Path(config.root_dir), Path(config.reports_dir)])

        # 3. Build the config object
        data_analysis_config = DataAnalysisConfig(
            root_dir=Path(config.root_dir),
            input_data_dir=Path(config.input_data_dir),
            reports_dir=Path(config.reports_dir),
            target_variables=target_variables
        )

        return data_analysis_config

In [19]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from mlProject import logger
from mlProject.entity.config_entity import DataAnalysisConfig

class DataAnalysis:
    def __init__(self, config: DataAnalysisConfig):
        self.config = config
        # Set style for professional research graphs
        plt.style.use('seaborn-v0_8-whitegrid')
        sns.set_context("talk")

    def analyze_missing_values(self):
        """
        Dynamically scans the input directory for all station CSVs,
        calculates missing values strictly for the schema variables,
        and saves a unified JSON report.
        """
        input_dir = Path(self.config.input_data_dir)
        report = {}

        # Scan the directory for any CSV files (no hardcoded station names!)
        csv_files = list(input_dir.glob("*_daily.csv"))
        
        if not csv_files:
            logger.warning(f"No CSV data found in {input_dir}")
            return

        for file_path in csv_files:
            station_name = file_path.stem.replace("_daily", "")
            logger.info(f"Analyzing missing data for station: {station_name}...")
            
            try:
                # Load the dataset
                df = pd.read_csv(file_path, index_col=0, parse_dates=True)
                
                # Filter for variables that exist in BOTH the schema and the CSV
                available_vars = [var for var in self.config.target_variables if var in df.columns]
                
                # Identify if any schema columns are completely missing from the station
                missing_vars = [var for var in self.config.target_variables if var not in df.columns]
                if missing_vars:
                    logger.warning(f"Station {station_name} is completely missing schema columns: {missing_vars}")

                # Calculate missing values
                null_counts = df[available_vars].isnull().sum().to_dict()
                total_rows = len(df)
                
                # Add to the master report
                report[station_name] = {
                    "total_rows": total_rows,
                    "null_counts": null_counts,
                    "missing_schema_columns": missing_vars
                }
                
            except Exception as e:
                logger.error(f"Failed analyzing data for {station_name}: {e}")
                raise e

        # Save the final report as a JSON file in the reports directory
        report_path = Path(self.config.reports_dir) / "missing_values_report.json"
        
        with open(report_path, "w") as f:
            json.dump(report, f, indent=4)
            
        logger.info(f"Successfully saved missing values report to {report_path}")

    def generate_research_graphs(self):
        """
        Generates visualizations of the raw data problems (NaNs, GPS gaps, non-linearities)
        to be used in the research paper, complete with professional annotations.
        """
        logger.info("Generating research graphs for data analysis...")
        
        # We will use KAN_L as the representative station for the graphs
        file_path = Path(self.config.input_data_dir) / "KAN_L_daily.csv"
        
        if not file_path.exists():
            logger.warning(f"Could not find {file_path} for graphing. Skipping graphs.")
            return

        df_raw = pd.read_csv(file_path, parse_dates=['time'])
        
        # ==========================================
        # GRAPH 1: Missing Sensors (The Math Crash Risk)
        # ==========================================
        plt.figure(figsize=(12, 5))
        window = df_raw[(df_raw['time'] > '2010-01-01') & (df_raw['time'] < '2012-01-01')]
        
        plt.plot(window['time'], window['t_u'], color='red', marker='.', linestyle='', alpha=0.7)

        window_nans = window[window['t_u'].isna()]
        if not window_nans.empty:
            start_gap = window_nans['time'].iloc[0]
            end_gap = window_nans['time'].iloc[-1]
            plt.axvspan(start_gap, end_gap, color='gray', alpha=0.2)

        plt.title("Raw Air Temperature ($T_u$) - Discontinuous Inputs", fontsize=14, pad=15)
        plt.xlabel("Date", fontsize=12)
        plt.ylabel("Air Temp (°C)", fontsize=12)
        plt.tight_layout()
        plt.savefig(Path(self.config.reports_dir) / "analysis_missing_sensors.png", dpi=300)
        plt.close()

        # ==========================================
        # GRAPH 2: Broken GPS (The Teleportation Bug)
        # ==========================================
        if 'gps_alt' in df_raw.columns:
            plt.figure(figsize=(12, 5))
            plt.plot(df_raw['time'], df_raw['gps_alt'], color='orange', linewidth=2)
            
            # Dynamically find the biggest jump/drop in altitude to point an arrow at it
            altitude_diffs = df_raw['gps_alt'].diff().abs()
            if not altitude_diffs.isna().all():
                max_jump_idx = altitude_diffs.idxmax()
                jump_time = df_raw.loc[max_jump_idx, 'time']
                jump_val = df_raw.loc[max_jump_idx, 'gps_alt']
                
            plt.title("Raw GPS Altitude - Spatial Discontinuity", fontsize=14, pad=15)
            plt.xlabel("Date", fontsize=12)
            plt.ylabel("Altitude (meters)", fontsize=12)
            plt.tight_layout()
            plt.savefig(Path(self.config.reports_dir) / "analysis_broken_gps.png", dpi=300)
            plt.close()

        # ==========================================
        # GRAPH 3: The 0°C Ceiling (Non-Linearity)
        # ==========================================
        if 't_u' in df_raw.columns and 't_surf' in df_raw.columns:
            plt.figure(figsize=(9, 7))
            sns.scatterplot(data=df_raw, x='t_u', y='t_surf', alpha=0.3, color='purple', edgecolor=None)
            
            # The 0-degree plateau line
            plt.axhline(0, color='red', linestyle='--', linewidth=2.5, label='0°C Melting Point')
            
            

            plt.title("Thermodynamic Non-Linearity: Air vs. Surface Temp", fontsize=14, pad=15)
            plt.xlabel("Air Temperature ($T_u$) [°C]", fontsize=12)
            plt.ylabel("Ice Surface Temperature ($T_{surf}$) [°C]", fontsize=12)
            plt.legend(loc='lower right', fontsize=11)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(Path(self.config.reports_dir) / "analysis_melting_plateau.png", dpi=300)
            plt.close()
            
        logger.info(f"Successfully generated research-grade annotated graphs in {self.config.reports_dir}")

In [20]:
try:
    logger.info(">>>>>> stage Data Analysis started <<<<<<")
    config = ConfigurationManager()
    data_analysis_config = config.get_data_analysis_config()
    
    data_analysis = DataAnalysis(config=data_analysis_config)
    
    # 1. Run your existing JSON report generator
    data_analysis.analyze_missing_values()
    
    # 2. Run the new graph generator for your paper
    data_analysis.generate_research_graphs()
    
    logger.info(">>>>>> stage Data Analysis completed <<<<<<\n\nx==========x")
except Exception as e:
    logger.exception(e)
    raise e

[2026-03-15 00:03:42,910: INFO: 518572082: >>>>>> stage Data Analysis started <<<<<<]
[2026-03-15 00:03:42,913: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-15 00:03:42,915: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-15 00:03:42,917: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-15 00:03:42,917: INFO: common: created directory at: artifacts]
[2026-03-15 00:03:42,917: INFO: common: created directory at: artifacts\data_analysis]
[2026-03-15 00:03:42,922: INFO: common: created directory at: artifacts\data_analysis\reports]
[2026-03-15 00:03:42,923: INFO: 3380827953: Analyzing missing data for station: EGP...]
[2026-03-15 00:03:42,962: INFO: 3380827953: Analyzing missing data for station: KAN_L...]
[2026-03-15 00:03:43,023: INFO: 3380827953: Analyzing missing data for station: KAN_M...]
[2026-03-15 00:03:43,088: INFO: 3380827953: Successfully saved missing values report to artifacts\data_analysis\reports\missing_valu

### Station 1: The Ablation Zone (e.g., KAN_L or SCO_L)

* These are "Lower" elevation stations near the coast. They experience intense summer melting where the surface temperature (t_surf) hits exactly 0°C and stays there while the ice turns to water. Your model must learn this 0°C physics ceiling.

In [7]:
from artifacts.data_ingestion.daily_data import *
import pandas as pd
from pathlib import Path

file_path1 = Path("artifacts/data_ingestion/daily_data/KAN_L_daily.csv")

# Load the dataset
cv1 = pd.read_csv(file_path1, index_col=0, parse_dates=True)

target_variables = [
    'lat', 'lon', 'alt',                           # Space
    't_a_u', 't_a_l',                              # Air Temperature
    'rh_u', 'rh_l',                                # Humidity
    'rainfall_cor_u', 'rainfall_cor_l',            # Precipitation
    'albedo',                                      # Surface Albedo
    't_surf'                                       # Ice Surface Temp
]

# 3. Filter for variables that actually exist in this specific station's dataset
# (This prevents a KeyError if a station is missing a specific sensor)
available_vars = [var for var in target_variables if var in cv1.columns]

# 4. Count the total null (missing) values for just these columns
null_counts = cv1[available_vars].isnull().sum()

total_rows = len(cv1)
print(f"Total rows in the entire dataset: {total_rows}\n")

print(f"--- Null Value Counts for Selected Variables in {file_path1.stem} ---")
print(null_counts.to_string())

Total rows in the entire dataset: 5838

--- Null Value Counts for Selected Variables in KAN_L_daily ---
rh_u       352
albedo    2647
t_surf     179


### Station 2: The Accumulation Zone (e.g., EGP or Summit)

* These are high-elevation stations deep in the interior. They rarely, if ever, see 0°C. The physics here are dominated by deep-freeze dry snow and extreme polar nights.

In [8]:
file_path2 = Path("artifacts/data_ingestion/daily_data/EGP_daily.csv")
cv2 = pd.read_csv(file_path2, index_col=0, parse_dates=True)

available_vars = [var for var in target_variables if var in cv2.columns]

# 4. Count the total null (missing) values for just these columns
null_counts = cv2[available_vars].isnull().sum()

total_rows = len(cv2)
print(f"Total rows in the entire dataset: {total_rows}\n")

print(f"--- Null Value Counts for Selected Variables in {file_path2.stem} ---")
print(null_counts.to_string())

Total rows in the entire dataset: 3601

--- Null Value Counts for Selected Variables in EGP_daily ---
rh_u                 3
rh_l              2969
rainfall_cor_u    3249
rainfall_cor_l    3601
albedo            2098
t_surf             907


### Handling Missing Values

**1. `rh_u` (Relative Humidity)**

* First, fallback to the lower boom backup sensor (`rh_l`). For any remaining short gaps (e.g., from sensor freezing), apply a spline interpolation with a strict 7-day limit.

**2. `albedo` (Surface Reflectivity)**

* Fill missing values with a physical constant of `0.85`. These gaps are caused by the dark polar night (no sunlight to measure), and 0.85 is the established albedo for fresh winter snow.

**3. `t_surface` (Ice Surface Temperature)**

* Leave as `NaN`. Since this is the target variable, we will isolate these missing rows and pass their Time and Space coordinates to the PINN as **Collocation Points**. The neural network will calculate these missing temperatures itself using the underlying physics equations.